In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor #or any model of your choice
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.metrics import make_scorer, mean_squared_error
#To get the root mean squared error
'''
rmse=mean_squared_error(y_true,y_pred,squared=False)
'''


'\nrmse=mean_squared_error(y_true,y_pred,squared=False)\n'

# Loading Dataset

In [2]:
try:
    train_df = pd.read_csv('./data/train.csv')
    print("Data loaded successfully!")
except FileNotFoundError:
    print("Error: The file 'car_prices.csv' was not found. Please check the file path.")

Data loaded successfully!


# Exploratory Data Analysis

In [3]:
def plots(columns,ncols_for_subplot, df):
    ncols = ncols_for_subplot
    nrows = -(-len(columns) // ncols)  # ceiling division

    fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(15, 5 * nrows))
    axes = axes.flatten()

    for i, col in enumerate(columns):
        sns.scatterplot(x=col, y='price', data=df, ax=axes[i])
        axes[i].set_title(f'Price vs. {col}')
        axes[i].set_xlabel(col)
        axes[i].set_ylabel('Price')

    # hide unused subplots if any
    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])

    plt.tight_layout()
    plt.show()

In [4]:
categorical_cols = train_df.select_dtypes(include='object').columns
numerical_cols = train_df.select_dtypes(exclude='object').columns
def exploratory_data_analysis(df):
    '''This function performs some preliminary EDA. You are free to add more to it to 
       guide you in preparing your dataset for trainiing
    '''
    print("First 5 rows of the dataset:")
    print(df.head())
    
    # Get information about the dataset (data types, non-null values)
    print("\nDataset information:")
    df.info()
    
    # Get descriptive statistics for numerical columns
    print("\nDescriptive statistics for numerical columns:")
    print(df.describe())
    #Get descriptive statistics for categorical columns
    print("\nDescriptive statistics for categorical columns:")
    print(df.describe(include='object'))
    #Checking for missing values
    print("\nMissing values per column:")
    print(df.isnull().sum())
    # Visualize the distribution of the target variable (price)
    plt.figure(figsize=(10, 6))
    sns.histplot(df['price'], kde=True, bins=50)
    plt.title('Distribution of Car Prices')
    plt.xlabel('Price')
    plt.ylabel('Frequency')
    plt.show()
    
    
    # Visualizing the relationship between all numerical features and price
    # For example, 'mileage' and 'price'
    print('Plotting numerical variables vs price')
    numerical_plot=plots(numerical_cols,2,df)
   

In [ ]:
exploratory_data_analysis(train_df)

First 5 rows of the dataset:
   id          brand              model  model_year  milage      fuel_type  \
0   0           MINI      Cooper S Base        2007  213000       Gasoline   
1   1        Lincoln              LS V8        2002  143250       Gasoline   
2   2      Chevrolet  Silverado 2500 LT        2002  136731  E85 Flex Fuel   
3   3        Genesis   G90 5.0 Ultimate        2017   19500       Gasoline   
4   4  Mercedes-Benz        Metris Base        2021    7388       Gasoline   

                                              engine  \
0       172.0HP 1.6L 4 Cylinder Engine Gasoline Fuel   
1       252.0HP 3.9L 8 Cylinder Engine Gasoline Fuel   
2  320.0HP 5.3L 8 Cylinder Engine Flex Fuel Capab...   
3       420.0HP 5.0L 8 Cylinder Engine Gasoline Fuel   
4       208.0HP 2.0L 4 Cylinder Engine Gasoline Fuel   

                     transmission ext_col int_col  \
0                             A/T  Yellow    Gray   
1                             A/T  Silver   Beige   
2     

In [ ]:
def filling_missing_values_in_numerical_columns(df):
    numerical_cols_with_missing = ['model_year', 'milage'] # Replace with your columns

    # Fill missing values using the median of each column
    for col in numerical_cols_with_missing:
        median_value = df[col].median()
        df[col].fillna(median_value, inplace=True)
    
    print("DataFrame after filling numerical missing values:")
    print(df.info())
    return df


In [ ]:
def filling_missing_values_in_categorical_columns(df):
        categorical_cols_with_missing = ['model', 'fuel_type'] # Replace with your columns
        # You could fill missing values using the mode of each column. Feel free to choose your strategy
        for col in categorical_cols_with_missing:
            mode_value = df[col].mode()[0]
            df[col].fillna(mode_value, inplace=True)
        
        print("DataFrame after filling categorical missing values:")
        print(df.info())
        return df

In [ ]:

def encode_categorical_columns(df):
    le = LabelEncoder()
    
   
    #categorical_cols = df.select_dtypes(include='object').columns
    print('Encoding variables...')
    for col in categorical_cols:
        
        try:
            if df[col].isnull().any():
                # If so, fill them with a placeholder string 'missing' before encoding
                df[col].fillna('missing', inplace=True)
            
            
            df[col] = le.fit_transform(df[col])
            print(f"Successfully applied Label Encoding to: {col}")
        except Exception as e:
            print(f"Could not apply Label Encoding to {col}. Error: {e}")
    return df

In [ ]:
def preprocessing(df):
    '''
    This function cleans your data for you. Feel free to tweak it to your tastes
    It comprises filling missing data, encoding categorical variables etc
    Perhaps feature engineering as well. Have fun!
    Argument:
        data (pd.DataFrame): The input DataFrame.
    
    Returns:
        pd.DataFrame: The preprocessed DataFrame.
    '''
    df=filling_missing_values_in_numerical_columns(df)
    df=filling_missing_values_in_categorical_columns(df)
    df=encode_categorical_columns(df)
    

    return df

In [ ]:
df=preprocessing(train_df)

In [ ]:
X = df.drop('price', axis=1) # Replace 'price' with your target column name
y = df['price']

# Split the data into training and testing sets (80% train, 20% test)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"\nTraining set size: {len(X_train)}")
print(f"Testing set size: {len(X_val)}")

In [ ]:
print(X_train.head())

# Training your Model

In [ ]:
def train_with_cv(estimator, param_grid, X, y, cv_splits=5, scoring=None):
    """
    Perform K-Fold Cross Validation with hyperparameter tuning. Adjust params as you please

    Parameters:
    estimator : sklearn estimator
        The model to train (e.g., RandomForestRegressor(), LogisticRegression(), etc.)
    param_grid : dict
        Hyperparameter search space, e.g., {'n_estimators': [100, 200], 'max_depth': [5, 10]}
    X : Features
    y : Target variable
    cv_splits : int
        Number of folds for cross-validation
    scoring : str or callable
        Scoring metric (default: neg_root_mean_squared_error for regression)
    """
    # default scoring = RMSE for regression
    if scoring is None:
        scoring = make_scorer(mean_squared_error, squared=False)

    kfold = KFold(n_splits=cv_splits, shuffle=True, random_state=42)

    grid_search = GridSearchCV(
        estimator=estimator,
        param_grid=param_grid,
        cv=kfold,
        scoring=scoring,
        n_jobs=-1,
        verbose=1
    )

    grid_search.fit(X, y)

    return grid_search.best_estimator_, grid_search.best_params_, grid_search.cv_results_

In [ ]:
model=RandomForestRegressor()
#model.fit(X_train,y_train)
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [5, 10, None],
    'min_samples_split': [2, 5]
}

best_model, best_params, cv_results = train_with_cv(model, param_grid, X_train, y_train)

print("Best Params:", best_params)

In [ ]:
y_val_pred = best_model.predict(X_val)

val_rmse = mean_squared_error(y_val, y_val_pred, squared=False)

print("Validation RMSE:", val_rmse)

In [ ]:
final_model = RandomForestRegressor(**best_params,random_state=42)

final_model.fit(
    np.vstack([X_train, X_val]), 
    np.hstack([y_train, y_val])
)



In [ ]:
test=pd.read_csv("./data/test.csv")
test.head()

In [ ]:
X_test=preprocessing(test)
X_test.head()

In [ ]:
y_test_pred = final_model.predict(X_test)
y_test_pred[0:5]

In [ ]:
submission=pd.DataFrame({"id":X_test['id'],"Price":y_test_pred})
submission.to_csv("Submission.csv",index=False)